# Vehicle Compliance Monitor
## AI-Powered Vehicle Compliance Checking System

This notebook processes dashcam videos to:
- Detect vehicles using YOLOv8
- Read license plates with OCR
- Check plates against compliance database
- Generate violation reports
- Create traffic analytics (counts, heatmaps, time profiles)

**Challenge:** AI Traffic Insights Challenge - Tallinn City Videos

## 1. Setup - Install Dependencies

This cell installs all required packages. Run this first!

In [ ]:
%%capture
# Install required packages
!pip install ultralytics opencv-python-headless easyocr torch torchvision
!pip install pandas numpy matplotlib seaborn tqdm Pillow

print("✓ All dependencies installed!")

## 2. Check GPU Availability

GPU acceleration significantly speeds up processing!

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✓ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ No GPU available, using CPU (will be slower)")
    print("  To enable GPU: Runtime → Change runtime type → GPU")

## 3. Upload Project Files

Upload your project files (src/ folder and data/)

In [ ]:
from google.colab import files
import zipfile
import os

print("Please upload your project ZIP file (containing src/ and data/ folders)")
print("Or clone from GitHub if you have a repository\n")

# Option 1: Upload ZIP file
uploaded = files.upload()

# Extract ZIP
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("✓ Files extracted")

# Option 2: Clone from GitHub (uncomment if using)
# !git clone https://github.com/your-username/vehicle-compliance-monitor.git
# %cd vehicle-compliance-monitor

# Verify structure
if os.path.exists('src') and os.path.exists('data'):
    print("\n✓ Project structure verified")
    !ls -la
else:
    print("\n⚠ Warning: src/ or data/ folder not found")
    print("Please ensure your ZIP contains the correct structure")

## 4. Upload Video Files

Upload your dashcam videos (Clip1_morning.mp4, Clip2_day.mp4)

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

# Create videos directory if it doesn't exist
Path('data/videos').mkdir(parents=True, exist_ok=True)

print("Please upload your video files (MP4, AVI, or MOV)\n")
uploaded_videos = files.upload()

# Move videos to data/videos/
for filename in uploaded_videos.keys():
    shutil.move(filename, f'data/videos/{filename}')
    print(f"✓ Moved {filename} to data/videos/")

# List uploaded videos
print("\nVideos ready for processing:")
!ls -lh data/videos/*.mp4 data/videos/*.avi data/videos/*.mov 2>/dev/null || echo "No videos found"

## 5. Configuration

Set processing parameters

In [ ]:
# Add src to Python path
import sys
sys.path.insert(0, 'src')

from models import PipelineConfig

# Create configuration
config = PipelineConfig(
    vehicle_confidence_threshold=0.5,  # Adjust if needed
    plate_confidence_threshold=0.6,
    ocr_confidence_threshold=0.7,
    frame_extraction_fps=5,  # Process 5 frames per second
    database_path='data/watchlist.csv',
    output_dir='outputs',
    enable_analytics=True  # Generate traffic analytics
)

print("Configuration:")
print(f"  Vehicle confidence: {config.vehicle_confidence_threshold}")
print(f"  Plate confidence: {config.plate_confidence_threshold}")
print(f"  OCR confidence: {config.ocr_confidence_threshold}")
print(f"  Frame extraction FPS: {config.frame_extraction_fps}")
print(f"  Database: {config.database_path}")
print(f"  Output directory: {config.output_dir}")
print(f"  Analytics: {'Enabled' if config.enable_analytics else 'Disabled'}")

## 6. Initialize Pipeline

Load all AI models and components

In [ ]:
import logging
from pipeline import Pipeline

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print("Initializing pipeline...")
print("This may take a minute as models are downloaded...\n")

pipeline = Pipeline(config)

print("\n✓ Pipeline initialized and ready!")

## 7. Process Videos

Run the complete pipeline on your videos

In [ ]:
import glob

# Find all videos
video_files = glob.glob('data/videos/*.mp4') + glob.glob('data/videos/*.avi') + glob.glob('data/videos/*.mov')

if not video_files:
    print("⚠ No video files found in data/videos/")
    print("Please upload videos in the previous cell")
else:
    print(f"Found {len(video_files)} video(s) to process:\n")
    for v in video_files:
        print(f"  - {v}")
    print("\n" + "="*70)
    print("STARTING PROCESSING")
    print("="*70 + "\n")
    
    # Process videos
    if len(video_files) == 1:
        results = pipeline.process_video(video_files[0])
    else:
        results = pipeline.process_batch(video_files)
    
    print("\n" + "="*70)
    print("✓ PROCESSING COMPLETE!")
    print("="*70)

## 8. View Results

Display generated visualizations and statistics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

# Display violation reports
print("=" * 70)
print("VIOLATION REPORTS")
print("=" * 70 + "\n")

csv_files = glob.glob('outputs/violations_*.csv')
for csv_file in csv_files:
    print(f"\n{csv_file}:")
    df = pd.read_csv(csv_file)
    if len(df) > 0:
        display(df)
    else:
        print("  No violations detected")

# Display summary reports
print("\n" + "=" * 70)
print("SUMMARY REPORTS")
print("=" * 70 + "\n")

summary_files = glob.glob('outputs/summary_*.txt')
for summary_file in summary_files:
    print(f"\n{summary_file}:")
    with open(summary_file, 'r') as f:
        print(f.read())

# Display visualizations
print("\n" + "=" * 70)
print("VISUALIZATIONS")
print("=" * 70 + "\n")

image_files = glob.glob('outputs/*.png')
for img_file in image_files:
    print(f"\n{img_file}:")
    display(Image(filename=img_file))

## 9. Download Results

Download all outputs as a ZIP file

In [ ]:
from google.colab import files
import shutil
from datetime import datetime

# Create ZIP of outputs
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_filename = f'vehicle_compliance_results_{timestamp}'

print("Creating ZIP file of all outputs...")
shutil.make_archive(zip_filename, 'zip', 'outputs')

print(f"\n✓ ZIP file created: {zip_filename}.zip")
print("Downloading...\n")

files.download(f'{zip_filename}.zip')

print("✓ Download complete!")
print("\nThe ZIP contains:")
print("  - Violation reports (CSV, JSON, TXT)")
print("  - Analytics visualizations (PNG)")
print("  - Time profiles and heatmaps")
print("  - Video comparison (if multiple videos)")
print("  - Annotated videos (if violations found)")

## 10. View Time Profiles

Analyze traffic patterns over time

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob

time_profile_files = glob.glob('outputs/time_profile_*.csv')

for tp_file in time_profile_files:
    print(f"\nTime Profile: {tp_file}")
    df = pd.read_csv(tp_file)
    display(df.head(10))
    
    print(f"\nTotal time intervals: {len(df)}")
    print(f"Total objects detected: {df['total'].sum() if 'total' in df.columns else 'N/A'}")

## 11. Compare Videos (if multiple processed)

View comparative statistics between morning and daytime videos

In [ ]:
import pandas as pd
from pathlib import Path

comparison_file = 'outputs/video_comparison.csv'

if Path(comparison_file).exists():
    print("VIDEO COMPARISON")
    print("=" * 70 + "\n")
    
    df = pd.read_csv(comparison_file)
    display(df)
    
    print("\nKey Insights:")
    print("  - Compare object counts between videos")
    print("  - Identify differences in traffic patterns")
    print("  - Analyze morning vs daytime variations")
else:
    print("No comparison file found (requires 2+ videos)")

## Summary

### What This Notebook Does:

1. **Vehicle Detection** - Uses YOLOv8 to detect cars, trucks, buses, motorcycles
2. **License Plate Reading** - OCR extracts plate numbers
3. **Compliance Checking** - Matches plates against watchlist database
4. **Violation Reporting** - Generates detailed reports in multiple formats
5. **Traffic Analytics** - Creates visualizations and statistics

### Outputs Generated:

- **Violation Reports**: CSV, JSON, and text summaries
- **Analytics**: Object counts, time profiles, heatmaps
- **Visualizations**: Professional plots and charts
- **Annotated Videos**: Videos with bounding boxes around violations
- **Comparisons**: Morning vs daytime analysis

### Challenge Requirements Met:

✓ Object detection with counts per class  
✓ Time profile (objects per 5 seconds)  
✓ Spatial pattern (heatmap)  
✓ Compare morning vs daytime  
✓ License plate detection and OCR  
✓ Compliance checking against database  

---

**Vehicle Compliance Monitor** - AI Traffic Insights Challenge  
Tallinn City Videos Analysis